# 07 — Modèles d'arbres : Arbre de décision et Random Forest

## Objectif

Après avoir évalué les modèles linéaires et différentes familles de variables dérivées, cette étape vise à étudier les modèles basés sur les arbres de décision.

Contrairement à la régression logistique, qui apprend une frontière de décision linéaire, un arbre de décision partitionne récursivement l'espace des variables grâce à une succession de questions binaires. Il est ainsi capable de capturer des effets de seuil, des interactions entre variables et des relations non linéaires.

Les objectifs de ce notebook sont les suivants :

- évaluer les performances d'un Arbre de Décision ;
- évaluer les performances d'une Random Forest ;
- optimiser les principaux hyperparamètres de la Random Forest grâce à un GridSearchCV respectant notre stratégie de validation temporelle ;
- comparer ces modèles aux références établies lors des étapes précédentes.

Toutes les expériences utilisent exactement la même validation temporelle (Expanding Window) afin de garantir une comparaison équitable entre les différents modèles.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent

if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

In [ ]:
import numpy as np
import pandas as pd
from src.data_loading import load_X_train, load_y_train
from sklearn.model_selection import GridSearchCV
from src.validation import create_expanding_window_folds, check_temporal_folds
from src.tree_models import build_decision_tree_pipeline, build_random_forest_pipeline
from src.evaluation import evaluate_model_on_folds, compare_model_results
from src.target import create_class_column

In [3]:
X_train = load_X_train()
y_train = load_y_train()

In [4]:
df_train , _ = create_class_column(X_train,y_train)

In [5]:
dates = sorted(list(set(X_train["TS"])))
folds = create_expanding_window_folds(dates)
check_temporal_folds(folds,dates,validation_size=120)

True

In [14]:
tuning_folds = folds[:-1]
final_holdout_fold = [folds[-1]]

tuning_end_date = tuning_folds[-1]["valid_end"]

df_tuning = df_train[
    df_train["TS"] <= tuning_end_date
].copy()

df_tuning = df_tuning.reset_index(drop=True)

In [15]:
def build_sklearn_temporal_splits(
    df: pd.DataFrame,
    folds: list[dict],
    time_col: str = "TS",
) -> list[tuple[np.ndarray, np.ndarray]]:
    """
    Convert date-based temporal folds into Scikit-Learn index splits.

    Parameters
    ----------
    df : pd.DataFrame
        Dataset used during hyperparameter search.

    folds : list[dict]
        Temporal fold definitions containing train and validation
        date boundaries.

    time_col : str, default="TS"
        Name of the temporal column.

    Returns
    -------
    list[tuple[np.ndarray, np.ndarray]]
        List containing one pair of train and validation indices
        for each temporal fold.
    """

    temporal_splits = []

    for fold in folds:
        train_mask = (
            (df[time_col] >= fold["train_start"])
            & (df[time_col] <= fold["train_end"])
        )

        valid_mask = (
            (df[time_col] >= fold["valid_start"])
            & (df[time_col] <= fold["valid_end"])
        )

        train_indices = np.flatnonzero(train_mask.to_numpy())
        valid_indices = np.flatnonzero(valid_mask.to_numpy())

        if len(train_indices) == 0 or len(valid_indices) == 0:
            raise ValueError(
                "A temporal GridSearchCV split contains an empty "
                "training or validation subset."
            )

        temporal_splits.append(
            (train_indices, valid_indices)
        )

    return temporal_splits

In [16]:
temporal_cv = build_sklearn_temporal_splits(
    df=df_tuning,
    folds=tuning_folds,
)

## Arbre de Décision

L'Arbre de Décision constitue le modèle le plus simple de la famille des modèles d'arbres.

Contrairement à la régression logistique, qui combine linéairement les variables explicatives, un arbre construit une succession de règles de décision basées sur des seuils optimaux.

L'objectif de cette première expérience est de déterminer si un modèle non linéaire est capable d'extraire davantage d'information prédictive à partir des rendements historiques utilisés jusqu'à présent.

In [6]:
tree = build_decision_tree_pipeline()
forest = build_random_forest_pipeline()

In [7]:
df_train.columns

Index(['ROW_ID', 'TS', 'ALLOCATION', 'RET_20', 'RET_19', 'RET_18', 'RET_17',
       'RET_16', 'RET_15', 'RET_14', 'RET_13', 'RET_12', 'RET_11', 'RET_10',
       'RET_9', 'RET_8', 'RET_7', 'RET_6', 'RET_5', 'RET_4', 'RET_3', 'RET_2',
       'RET_1', 'SIGNED_VOLUME_20', 'SIGNED_VOLUME_19', 'SIGNED_VOLUME_18',
       'SIGNED_VOLUME_17', 'SIGNED_VOLUME_16', 'SIGNED_VOLUME_15',
       'SIGNED_VOLUME_14', 'SIGNED_VOLUME_13', 'SIGNED_VOLUME_12',
       'SIGNED_VOLUME_11', 'SIGNED_VOLUME_10', 'SIGNED_VOLUME_9',
       'SIGNED_VOLUME_8', 'SIGNED_VOLUME_7', 'SIGNED_VOLUME_6',
       'SIGNED_VOLUME_5', 'SIGNED_VOLUME_4', 'SIGNED_VOLUME_3',
       'SIGNED_VOLUME_2', 'SIGNED_VOLUME_1', 'MEDIAN_DAILY_TURNOVER', 'GROUP',
       'target', 'class'],
      dtype='str')

In [8]:
features = df_train.drop(
    columns=['ROW_ID', 'TS', 'ALLOCATION', 'target', 'class']
).columns

In [17]:
X_tuning = df_tuning[features]
y_tuning = df_tuning["class"]

In [9]:
tree_results = evaluate_model_on_folds(df_train,folds,tree,features)

In [10]:
tree_results

,fold,train_start,train_end,valid_start,valid_end,train_positive_rate,valid_positive_rate,accuracy,log_loss,roc_auc,n_valid_predictions,n_correct_predictions
0,1,DATE_0001,DATE_2042,DATE_2043,DATE_2162,0.504803,0.522228,0.517459,0.693084,0.515110,24114,12478
1,2,DATE_0001,DATE_2162,DATE_2163,DATE_2282,0.505732,0.504837,0.529759,0.690758,0.544156,24396,12924
2,3,DATE_0001,DATE_2282,DATE_2283,DATE_2402,0.505687,0.520554,0.526732,0.691817,0.526618,24764,13044
3,4,DATE_0001,DATE_2402,DATE_2403,DATE_2522,0.506421,0.522097,0.521512,0.691873,0.523377,25660,13382


### Interprétation

Les performances obtenues par l'Arbre de Décision restent proches de celles de la régression logistique.

Bien que ce modèle soit capable de capturer des relations non linéaires et des effets de seuil, ces propriétés ne se traduisent pas par une amélioration significative des performances sur les différentes fenêtres de validation temporelle.

Ce résultat est cohérent avec la théorie : un arbre de décision possède une variance élevée et a donc tendance à sur-apprendre les données d'entraînement, ce qui limite sa capacité de généralisation sur des périodes de marché jamais observées.

## Random Forest

La Random Forest est une méthode d'ensemble reposant sur le principe du Bagging.

Elle entraîne un grand nombre d'arbres de décision sur des échantillons bootstrap différents et introduit une sélection aléatoire des variables à chaque division.

L'objectif est de réduire la variance d'un arbre de décision tout en conservant sa capacité à modéliser des relations complexes.

L'hypothèse testée dans cette partie est qu'une forêt composée de nombreux arbres diversifiés généralisera mieux qu'un arbre unique.

In [11]:
forest_results = evaluate_model_on_folds(df_train,folds,forest,features)

In [12]:
forest_results

,fold,train_start,train_end,valid_start,valid_end,train_positive_rate,valid_positive_rate,accuracy,log_loss,roc_auc,n_valid_predictions,n_correct_predictions
0,1,DATE_0001,DATE_2042,DATE_2043,DATE_2162,0.504803,0.522228,0.518786,0.692042,0.521718,24114,12510
1,2,DATE_0001,DATE_2162,DATE_2163,DATE_2282,0.505732,0.504837,0.538449,0.690453,0.551737,24396,13136
2,3,DATE_0001,DATE_2282,DATE_2283,DATE_2402,0.505687,0.520554,0.525481,0.691744,0.526864,24764,13013
3,4,DATE_0001,DATE_2402,DATE_2403,DATE_2522,0.506421,0.522097,0.524006,0.691468,0.527794,25660,13446


### Interprétation

La Random Forest obtient des performances légèrement supérieures à celles de l'Arbre de Décision sur la majorité des métriques étudiées.

Cette amélioration confirme le rôle du Bagging : l'agrégation de plusieurs arbres permet de réduire la variance des prédictions et d'améliorer la capacité de généralisation du modèle.

Cependant, les gains restent relativement modestes par rapport à la régression logistique, ce qui suggère que la principale limite provient probablement davantage de l'information contenue dans les variables que du modèle lui-même.

## Optimisation des hyperparamètres

Après avoir construit une première Random Forest, nous cherchons à déterminer si ses performances peuvent être améliorées grâce à une optimisation de ses principaux hyperparamètres.

Contrairement au GridSearchCV classique proposé par Scikit-Learn, cette recherche utilise exclusivement notre stratégie de validation temporelle développée lors des étapes précédentes.

Cette approche garantit l'absence de fuite temporelle et permet de conserver un protocole expérimental identique à celui utilisé pour l'ensemble des modèles évalués dans ce projet.

In [18]:
forest_for_search = build_random_forest_pipeline(
    n_estimators=50,
    bootstrap=True,
    oob_score=False,
    n_jobs=1,
    random_state=42,
)

In [20]:
param_grid = {
    "classifier__max_depth": [5, 8, 12],
    "classifier__min_samples_leaf": [10, 20],
    "classifier__max_features": ["sqrt", 0.5],
}

In [21]:
scoring = {
    "accuracy": "accuracy",
    "roc_auc": "roc_auc",
    "neg_log_loss": "neg_log_loss",
}

grid_search = GridSearchCV(
    estimator=forest_for_search,
    param_grid=param_grid,
    scoring=scoring,
    refit="accuracy",
    cv=temporal_cv,
    n_jobs=-1,
    verbose=2,
    return_train_score=True,
)

grid_search.fit(
    X_tuning,
    y_tuning,
)

Fitting 3 folds for each of 12 candidates, totalling 36 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'classifier__max_depth': [5, 8, ...], 'classifier__max_features': ['sqrt', 0.5], 'classifier__min_samples_leaf': [10, 20]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.","{'accuracy': 'accuracy', 'neg_log_loss': 'neg_log_loss', 'roc_auc': 'roc_auc'}"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",'accuracy'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.","[(array([ 0...ape=(428139,)), ...), (array([ 0...a

In [23]:
grid_search.best_params_

{'classifier__max_depth': 8,
 'classifier__max_features': 0.5,
 'classifier__min_samples_leaf': 20}

In [24]:
grid_search.best_score_

np.float64(0.5278993265040975)

In [25]:
grid_results = pd.DataFrame(
    grid_search.cv_results_
)

grid_summary = grid_results[
    [
        "param_classifier__max_depth",
        "param_classifier__min_samples_leaf",
        "param_classifier__max_features",
        "mean_train_accuracy",
        "mean_test_accuracy",
        "std_test_accuracy",
        "mean_test_roc_auc",
        "mean_test_neg_log_loss",
        "mean_fit_time",
        "rank_test_accuracy",
    ]
].copy()

grid_summary["mean_test_log_loss"] = (
    -grid_summary["mean_test_neg_log_loss"]
)

grid_summary = grid_summary.drop(
    columns="mean_test_neg_log_loss"
)

grid_summary = grid_summary.sort_values(
    by="rank_test_accuracy"
).reset_index(drop=True)

grid_summary

,param_classifier__max_depth,param_classifier__min_samples_leaf,param_classifier__max_features,mean_train_accuracy,mean_test_accuracy,std_test_accuracy,mean_test_roc_auc,mean_fit_time,rank_test_accuracy,mean_test_log_loss
0,8,20,0.5,0.549635,0.527899,0.006046,0.534165,4093.070215,1,0.691062
1,5,20,sqrt,0.530226,0.526993,0.006505,0.531538,398.260014,2,0.691767
2,8,10,0.5,0.550113,0.526989,0.007009,0.533747,4133.281255,3,0.691126
3,8,20,sqrt,0.547342,0.526736,0.007919,0.531475,609.162121,4,0.691533
4,5,10,0.5,0.529421,0.526540,0.007681,0.533172,1331.306562,5,0.691258
5,5,10,sqrt,0.530479,0.526418,0.006862,0.531195,388.906331,6,0.691790
6,5,20,0.5,0.529313,0.526319,0.007618,0.533270,1345.961624,7,0.691243
7,12,10,0.5,0.621596,0.525475,0.007724,0.530147,1794.906293,8,0.691555
8,8,10,sqrt,0.548175,0.525098,0.008685,0.531765,610.030650,9,0.691468
9,12,20,sqrt,0.598223,0.524508,0.007584,0.530010,3013.674823,10,0.691479


### Interprétation

Plusieurs combinaisons de profondeur maximale, de taille minimale des feuilles et de nombre de variables considérées à chaque division ont été évaluées.

Certaines configurations améliorent légèrement les performances moyennes obtenues sur les folds utilisés pendant la phase d'optimisation.

Cependant, lorsque le meilleur modèle est évalué sur une fenêtre temporelle indépendante, cette amélioration ne se confirme pas.

Ce résultat souligne l'importance de distinguer la sélection des hyperparamètres de l'évaluation finale du modèle afin d'obtenir une estimation réaliste de ses capacités de généralisation.

In [26]:
best_params = grid_search.best_params_

In [27]:
best_forest = build_random_forest_pipeline(
    n_estimators=200,
    max_depth=best_params[
        "classifier__max_depth"
    ],
    min_samples_leaf=best_params[
        "classifier__min_samples_leaf"
    ],
    max_features=best_params[
        "classifier__max_features"
    ],
    bootstrap=True,
    oob_score=False,
    n_jobs=-1,
    random_state=42,
)

### Interprétation

Le meilleur modèle sélectionné par le GridSearchCV n'améliore pas significativement les performances sur le fold de validation indépendant.

Malgré une légère amélioration observée durant la phase d'optimisation, cette progression ne se généralise pas sur des données totalement exclues du processus de sélection.

Cette observation confirme que l'optimisation des hyperparamètres ne permet pas, à elle seule, d'extraire davantage d'information prédictive à partir des variables actuellement disponibles.

In [28]:
best_forest_holdout_results = evaluate_model_on_folds(
    df=df_train,
    folds=final_holdout_fold,
    model=best_forest,
    feature_cols=features,
)

best_forest_holdout_results

,fold,train_start,train_end,valid_start,valid_end,train_positive_rate,valid_positive_rate,accuracy,log_loss,roc_auc,n_valid_predictions,n_correct_predictions
0,1,DATE_0001,DATE_2402,DATE_2403,DATE_2522,0.506421,0.522097,0.522292,0.691175,0.529409,25660,13402


# Conclusion

Cette étape avait pour objectif d'évaluer si les modèles d'arbres pouvaient mieux exploiter les rendements historiques que la régression logistique étudiée précédemment.

Les principales conclusions sont les suivantes :

- L'Arbre de Décision est capable de modéliser des relations non linéaires mais ne surpasse pas significativement la régression logistique.
- La Random Forest améliore légèrement les performances grâce au Bagging et à la réduction de la variance des arbres individuels.
- L'optimisation des hyperparamètres par GridSearchCV apporte des gains très limités qui ne se confirment pas sur une période de validation totalement indépendante.

Ces résultats suggèrent que, dans le contexte actuel du challenge QRT, la principale limite ne provient probablement plus du choix du modèle mais plutôt de la qualité et de la richesse des variables explicatives utilisées.

La prochaine étape du projet consistera à étudier les méthodes de Boosting (AdaBoost puis Gradient Boosting) afin d'évaluer si une stratégie d'apprentissage séquentielle permet d'améliorer davantage les performances prédictives.